In [ ]:
import pandas as pd

# Load the dataset
file_path = "EV_Charging_Stations_2_Feb82024.csv"
df = pd.read_csv(file_path, encoding="latin1", delimiter=",", quotechar='"', on_bad_lines="skip", low_memory=False)

# Quick preview
df.head()


In [72]:
# cleaning step 1

# Drop columns with mostly null values or unrelated to electric charging
cols_to_drop = [
    'Plus4', 'Expected Date', 'BD Blends', 'NG Fill Type Code', 'NG PSI',
    'Hydrogen Status Link', 'NG Vehicle Class', 'LPG Primary', 'E85 Blender Pump',
    'Intersection Directions (French)', 'Access Days Time (French)', 'BD Blends (French)',
    'Groups With Access Code (French)', 'CNG Dispenser Num', 'CNG On-Site Renewable Source',
    'CNG Total Compression Capacity', 'CNG Storage Capacity', 'LNG On-Site Renewable Source',
    'EV Pricing (French)', 'LPG Nozzle Types', 'Hydrogen Pressures', 'Hydrogen Standards',
    'CNG Fill Type Code', 'CNG PSI', 'CNG Vehicle Class', 'LNG Vehicle Class',
    'RD Blends', 'RD Blends (French)', 'RD Blended with Biodiesel',
    'RD Maximum Biodiesel Level', 'CNG Station Sells Renewable Natural Gas', 'Intersection Directions',
    'Station Phone', 'Cards Accepted', 'EV Other Info', 'Geocode Status', 'Owner Type Code',
    'Federal Agency ID', 'Federal Agency Name', 'Open Date', 'Hydrogen Is Retail',
    'Access Code', 'Access Detail Code', 'Federal Agency Code', 'Facility Type',
    'E85 Other Ethanol Blends', 'EV Pricing', 'EV On-Site Renewable Source',
    'Restricted Access', 'NPS Unit Name', 'LNG Station Sells Renewable Natural Gas',
    'Maximum Vehicle Class', 'EV Workplace Charging', 'EV Network Web',
    'Access Days Time' 


]
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')


In [ ]:
# Quick preview
df.head()

In [74]:
# cleaning step 2

df.rename(columns={
    'EV Level1 EVSE Num': 'Level1_Chargers',
    'EV Level2 EVSE Num': 'Level2_Chargers',
    'EV DC Fast Count': 'DC_Fast_Chargers',
    'EV Network': 'EV_Network',
    'EV Connector Types': 'Connector_Types'
}, inplace=True)


In [75]:
# cleaning step 3

# Keep only rows where State is a valid two-letter code
df = df[df['State'].notnull() & df['State'].str.match("^[A-Z]{2}$")]


In [76]:
# cleaning step 4 

#  Convert Charger Counts to Numeric
df['Level1_Chargers'] = pd.to_numeric(df['Level1_Chargers'], errors='coerce')
df['Level2_Chargers'] = pd.to_numeric(df['Level2_Chargers'], errors='coerce')
df['DC_Fast_Chargers'] = pd.to_numeric(df['DC_Fast_Chargers'], errors='coerce')


In [77]:
# cleaning step 5

# Remove Duplicate Stations
df.drop_duplicates(subset=['Station Name', 'Street Address', 'City', 'State'], inplace=True)


In [78]:
# cleaning step 6

# Drop rows missing long / lat locations
df = df[df['Latitude'].notnull() & df['Longitude'].notnull()]


In [79]:
# cleaning step 7

# fill missing charger counts with 0
df[['Level1_Chargers', 'Level2_Chargers', 'DC_Fast_Chargers']] = df[[
    'Level1_Chargers', 'Level2_Chargers', 'DC_Fast_Chargers'
]].fillna(0)


In [80]:
# Cleaning step 8

# create total chargers column
df['Total_Chargers'] = df['Level1_Chargers'] + df['Level2_Chargers'] + df['DC_Fast_Chargers']


In [81]:
# cleaning step 9 

# Standardize cases in names
df['EV_Network'] = df['EV_Network'].str.title()
df['City'] = df['City'].str.title()


In [82]:
# cleaning step 10

# Keep only electric vehicle charging stations
# dataset includes multiple types of alternative fuel stations
df = df[df["Fuel Type Code"] == "ELEC"]


In [ ]:
df.head(10)       # first 10 rows
df.tail(10)       # last 10 rows
df.sample(10)     # 10 random rows


In [ ]:
# Group by state and sum total chargers
chargers_by_state = df.groupby("State")["Total_Chargers"].sum().sort_values(ascending=False)

# Add a new column: total chargers in that state
df["Total_Chargers_in_State"] = df["State"].map(chargers_by_state)

# Display the result
chargers_by_state


In [85]:
df.to_csv("Cleaner_EV_Charging_Stations_3.csv", index=False)
print("Cleaned data saved!")


Cleaned data saved!
